In [ ]:
!pip install torch torchvision torchaudio
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import networkx as nx
import random
import zipfile
import os
import re
import requests

from torch_geometric.utils import from_networkx
from torch_geometric.nn import GCNConv
from google.colab import drive

In [ ]:

drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/BioGRID/BIOGRID-MV-Physical-LATEST.tab2.zip'
extract_path = '/content/biogrid_data'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    print(zip_ref.namelist())

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

txt_file = [f for f in os.listdir(extract_path) if f.endswith('.txt')][0]
full_file_path = os.path.join(extract_path, txt_file)
print(full_file_path)
print("Reading MITAB file...")
df = pd.read_csv(full_file_path, sep='\t', low_memory=False)
df.columns = [c.strip() for c in df.columns]

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)



print("\n===== FIRST 5 ROWS (FULL) =====\n")
print(df.head())

Mounted at /content/drive
['BIOGRID-MV-Physical-5.0.257.mitab.txt']
/content/biogrid_data/BIOGRID-MV-Physical-5.0.257.mitab.txt
Reading MITAB file...

===== FIRST 5 ROWS (FULL) =====

             #ID Interactor A              ID Interactor B  \
0  entrez gene/locuslink:6416   entrez gene/locuslink:2318   
1  entrez gene/locuslink:2624   entrez gene/locuslink:5371   
2  entrez gene/locuslink:6118   entrez gene/locuslink:6774   
3   entrez gene/locuslink:375  entrez gene/locuslink:23163   
4   entrez gene/locuslink:377  entrez gene/locuslink:23647   

                                                                                                                                                                                     Alt IDs Interactor A  \
0                                                                                              biogrid:112315|entrez gene/locuslink:MAP2K4|uniprot/swiss-prot:P45985|refseq:NP_003001|refseq:NP_001268364   
1                                

In [ ]:
#  HUMAN & SARS-CoV-2
HUMAN_TAXID = "9606"
VIRUS_TAXID = "2697049"

is_human_a = df['Taxid Interactor A'].str.contains(HUMAN_TAXID, na=False)
is_human_b = df['Taxid Interactor B'].str.contains(HUMAN_TAXID, na=False)

print("\n===== FIRST 5 ROWS (FULL) =====\n")
print(is_human_a.head())

is_virus_a = df['Taxid Interactor A'].str.contains(VIRUS_TAXID, na=False)
is_virus_b = df['Taxid Interactor B'].str.contains(VIRUS_TAXID, na=False)
print("\n===== FIRST 5 ROWS (FULL) =====\n")
print(is_virus_a.head())

virus_human_df = df[
    (is_human_a & is_virus_b) |
    (is_virus_a & is_human_b)
].copy()
print("\n===== FIRST 5 ROWS (FULL) =====\n")
print(virus_human_df.head())

print(f"Filtered interactions: {len(virus_human_df)}")




===== FIRST 5 ROWS (FULL) =====

0    True
1    True
2    True
3    True
4    True
Name: Taxid Interactor A, dtype: bool

===== FIRST 5 ROWS (FULL) =====

0    False
1    False
2    False
3    False
4    False
Name: Taxid Interactor A, dtype: bool

===== FIRST 5 ROWS (FULL) =====

                      #ID Interactor A               ID Interactor B  \
314227  entrez gene/locuslink:43740570   entrez gene/locuslink:23476   
314228  entrez gene/locuslink:43740570    entrez gene/locuslink:6046   
314229  entrez gene/locuslink:43740570   entrez gene/locuslink:10283   
314230  entrez gene/locuslink:43740570  entrez gene/locuslink:124245   
314231  entrez gene/locuslink:43740571    entrez gene/locuslink:9512   

                                                                                                          Alt IDs Interactor A  \
314227  biogrid:4383845|entrez gene/locuslink:E|entrez gene/locuslink:GU280_gp04|uniprot/swiss-prot:P0DTC4|refseq:YP_009724392   
314228  biogrid:4383845|

In [ ]:

def extract_uniprot(alt_id_string):
    if pd.isna(alt_id_string):
        return None
    match = re.search(r'uniprot(?:/swiss-prot)?:\s*([A-Z0-9]{6,10})', str(alt_id_string), re.IGNORECASE)
    return match.group(1) if match else None


virus_human_df['Uniprot_A'] = virus_human_df['Alt IDs Interactor A'].apply(extract_uniprot)
virus_human_df['Uniprot_B'] = virus_human_df['Alt IDs Interactor B'].apply(extract_uniprot)

virus_human_df = virus_human_df.dropna(subset=['Uniprot_A', 'Uniprot_B'])






In [ ]:

def order_pair(row):
    if "9606" in row['Taxid Interactor A']:
        return row['Uniprot_A'], row['Uniprot_B']
    else:
        return row['Uniprot_B'], row['Uniprot_A']

virus_human_df[['Human', 'Virus']] = virus_human_df.apply(
    order_pair, axis=1, result_type='expand'
)


sample_df = virus_human_df.head(700)

unique_protein_ids = set(sample_df['Human']).union(set(sample_df['Virus']))

print(f"Nodes: {len(unique_protein_ids)} | Edges: {len(sample_df)}")

def fetch_uniprot_sequences(unique_protein_ids):
    proteins_dict = {}
    batch_size = 25
    id_list = list(unique_protein_ids)
# https://rest.uniprot.org/uniprotkb/search?query=P0DTC4&fields=accession,sequence&size=1
    for i in range(0, len(id_list), batch_size):
        batch = id_list[i:i+batch_size]
        query = " OR ".join([f"accession:{pid}" for pid in batch])

        url = "https://rest.uniprot.org/uniprotkb/search"
        params = {'query': query, 'fields': 'accession,sequence', 'size': batch_size}

        response = requests.get(url, params=params)
        if response.status_code == 200:
            for record in response.json().get('results', []):
                acc = record.get('primaryAccession')
                seq = record.get('sequence', {}).get('value', '')
                if acc and seq:
                    proteins_dict[acc] = seq

    return proteins_dict

proteins = fetch_uniprot_sequences(unique_protein_ids)

sample_df = sample_df[
    sample_df['Human'].isin(proteins.keys()) &
    sample_df['Virus'].isin(proteins.keys())
]

edges = list(zip(sample_df['Human'], sample_df['Virus']))

random.shuffle(edges)

split_idx = int(0.8 * len(edges))

train_edges = edges[:split_idx]
test_edges  = edges[split_idx:]

print(f"Train edges: {len(train_edges)}")
print(f"Test edges: {len(test_edges)}")

node_list = list(proteins.keys())
node_index = {n: i for i, n in enumerate(node_list)}

Nodes: 598 | Edges: 700
Train edges: 558
Test edges: 140


In [ ]:
def autocovariance(seq, lag=10):
    aa_dict = {aa: i for i, aa in enumerate("ACDEFGHIKLMNPQRSTVWY")}
    values = [aa_dict[a] for a in seq if a in aa_dict]

    if len(values) < lag + 1:
        return [0.0] * lag

    mean = np.mean(values)
    ac = []

    for d in range(1, lag + 1):
        cov = np.mean([
            (values[i] - mean) * (values[i + d] - mean)
            for i in range(len(values) - d)
        ])
        ac.append(cov)

    return ac

In [ ]:

x = torch.tensor(
    [autocovariance(proteins[n]) for n in node_list],
    dtype=torch.float
)
print(x)
edge_index = torch.tensor(
    [[node_index[u], node_index[v]] for u, v in train_edges],
    dtype=torch.long
).t()

print("\n===== RAW EDGE_INDEX TENSOR for graph  =====\n")
print(edge_index[:, :20])

from torch_geometric.data import Data

data = Data(x=x, edge_index=edge_index)

print(data)
print("\n===== FIRST 20 EDGE INDICES =====\n")

for i in range(20):
    u = edge_index[0, i].item()
    v = edge_index[1, i].item()
    print(f"{i+1}. ({u}, {v})")

tensor([[-1.3914,  0.9206,  0.0231,  ...,  0.5716,  0.3392,  0.9749],
        [ 1.1846,  0.3312,  1.1638,  ..., -0.4828, -0.2011,  0.5173],
        [ 0.7846,  1.2981,  1.4114,  ..., -0.1484,  0.7020,  0.7416],
        ...,
        [ 1.5036, -0.5055,  0.4351,  ...,  0.8566, -0.6875, -0.8110],
        [-3.5635, -2.2390,  0.6528,  ...,  1.2133, -0.4961, -0.7750],
        [-1.8919, -2.5822, -0.3187,  ...,  0.9737, -2.0000,  0.6797]])

===== RAW EDGE_INDEX TENSOR for graph  =====

tensor([[ 84,  59, 235, 338,  16, 389, 287, 298,  57, 232, 507, 398, 586, 108,
         519, 138, 240, 247, 314, 117],
        [594,  20, 469, 397,  20, 422, 495,  20,  20,  20, 488, 594,  20,  20,
          20, 495, 594,  20, 594, 495]])
Data(x=[596, 10], edge_index=[2, 558])

===== FIRST 20 EDGE INDICES =====

1. (84, 594)
2. (59, 20)
3. (235, 469)
4. (338, 397)
5. (16, 20)
6. (389, 422)
7. (287, 495)
8. (298, 20)
9. (57, 20)
10. (232, 20)
11. (507, 488)
12. (398, 594)
13. (586, 20)
14. (108, 20)
15. (519, 20)
1

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(10, 64)
        self.conv2 = GCNConv(64, 64)
        self.fc = torch.nn.Linear(128, 1)

    def forward(self, data, edge_pairs):
        x = F.relu(self.conv1(data.x, data.edge_index))
        x = self.conv2(x, data.edge_index)

        u = x[edge_pairs[0]]
        v = x[edge_pairs[1]]

        return torch.sigmoid(self.fc(torch.cat([u, v], dim=1)))


model = GCN()
opt = torch.optim.Adam(model.parameters(), lr=0.01)




In [ ]:
train_edge_pairs = torch.tensor(
    [[node_index[u], node_index[v]] for u, v in train_edges],
    dtype=torch.long
).t()




print("\nTraining model...")

for epoch in range(101):
    model.train()
    opt.zero_grad()

    preds = model(data, train_edge_pairs)

    labels = torch.ones_like(preds)

    loss = F.binary_cross_entropy(preds, labels)

    loss.backward()
    opt.step()

    with torch.no_grad():
        predicted = (preds > 0.5).float()
        correct = (predicted == labels).sum().item()
        accuracy = correct / labels.size(0)

    if epoch % 20 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f} | Acc: {accuracy*100:.2f}%")



Training model...
Epoch 0 | Loss: 0.9002 | Acc: 7.35%
Epoch 20 | Loss: 0.0000 | Acc: 100.00%
Epoch 40 | Loss: 0.0000 | Acc: 100.00%
Epoch 60 | Loss: 0.0000 | Acc: 100.00%
Epoch 80 | Loss: 0.0000 | Acc: 100.00%
Epoch 100 | Loss: 0.0000 | Acc: 100.00%


In [ ]:
test_edge_pairs = torch.tensor(
    [[node_index[u], node_index[v]] for u, v in test_edges],
    dtype=torch.long
).t()

model.eval()

with torch.no_grad():

    preds = model(data, test_edge_pairs)

    labels = torch.ones_like(preds)

    predicted = (preds > 0.5).float()
    correct = (predicted == labels).sum().item()
    accuracy = correct / labels.size(0)

print(f"\nTest Accuracy: {accuracy*100:.2f}%")


Test Accuracy: 100.00%
